# Rereanking

In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser


In [2]:
##load

loader = TextLoader("sample.txt")
raw_docs = loader.load()

##textsplitter (split text into docs chunks)
text_splitter = RecursiveCharacterTextSplitter (chunk_size = 300, chunk_overlap = 50)
docs = text_splitter.split_documents(raw_docs)


In [3]:
docs

[Document(metadata={'source': 'sample.txt'}, page_content='BMW is a German luxury automobile manufacturer founded in 1916.\n\nBMW is headquartered in Munich, Germany.\n\nBMW produces premium cars, motorcycles, and electric vehicles.\n\nThe BMW i4 and BMW iX are popular electric vehicle models.'),
 Document(metadata={'source': 'sample.txt'}, page_content='BMW continues to invest heavily in sustainable transportation technologies.\n\nMercedes-Benz is another German automotive company known for luxury vehicles.\n\nMercedes-Benz develops advanced driver assistance and autonomous driving systems.\n\nThe Mercedes EQS is a flagship electric luxury sedan.'),
 Document(metadata={'source': 'sample.txt'}, page_content="Audi is a premium automobile manufacturer owned by the Volkswagen Group.\n\nAudi focuses on innovation, performance, and electric mobility.\n\nThe Audi e-tron series represents the company's electric vehicle lineup.\n\nTesla is an American electric vehicle company founded by Elon M

In [4]:
#create a query

query = "What is BMW ?"

In [5]:
## Fais and embedding model

from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

vectorestore = FAISS.from_documents(docs,embedding)

retriever = vectorestore.as_retriever(search_kwargs={"k":8})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11517.28it/s]


In [7]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# Load environment variables
load_dotenv()
import os

# Initialize LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="qwen/qwen3-32b"
)

# Invoke model
response = llm.invoke("Hi")
print(response)

content='<think>\nOkay, the user sent "Hi". I need to respond appropriately. Since it\'s a greeting, I should acknowledge it and offer help. Maybe say something like "Hello! How can I assist you today?" to keep the conversation open. Let me check if there are any other possible interpretations, but "Hi" is pretty straightforward as a greeting. I\'ll go with a friendly and helpful response.\n</think>\n\nHello! How can I assist you today? 😊' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 9, 'total_tokens': 105, 'completion_time': 0.193512375, 'prompt_time': 0.000407973, 'queue_time': 0.055086006, 'total_time': 0.193920348}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--019ea364-b845-77a3-9c6a-83c663518d9d-0' usage_metadata={'input_tokens': 9, 'output_tokens': 96, 'total_tokens': 105}


In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a document reranking assistant.

Query:
{question}

Documents:
{documents}
                                          

Instruction:

think about the user question 
Rank the documents from most relevant to least relevant.
Return only a Python list of document indices.

Example:
[2, 0, 1]
""")

In [9]:
retrieve_docs = retriever.invoke(query)
retrieve_docs

[Document(id='3151b61c-d7ef-487c-8a68-a7e3f9c2f166', metadata={'source': 'sample.txt'}, page_content='BMW is a German luxury automobile manufacturer founded in 1916.\n\nBMW is headquartered in Munich, Germany.\n\nBMW produces premium cars, motorcycles, and electric vehicles.\n\nThe BMW i4 and BMW iX are popular electric vehicle models.'),
 Document(id='618d3805-b05f-4d4f-8c37-a02e4c492adb', metadata={'source': 'sample.txt'}, page_content='BMW continues to invest heavily in sustainable transportation technologies.\n\nMercedes-Benz is another German automotive company known for luxury vehicles.\n\nMercedes-Benz develops advanced driver assistance and autonomous driving systems.\n\nThe Mercedes EQS is a flagship electric luxury sedan.'),
 Document(id='09e2e504-4d3f-4ae9-957d-0737d9c61b4e', metadata={'source': 'sample.txt'}, page_content="Audi is a premium automobile manufacturer owned by the Volkswagen Group.\n\nAudi focuses on innovation, performance, and electric mobility.\n\nThe Audi e

In [10]:
chain = prompt | llm | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a document reranking assistant.\n\nQuery:\n{question}\n\nDocuments:\n{documents}\n\n\nInstruction:\n\nthink about the user question \nRank the documents from most relevant to least relevant.\nReturn only a Python list of document indices.\n\nExample:\n[2, 0, 1]\n'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x122c5ab90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x13bc601d0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

In [11]:
doc_lines = [f" {i+1} {doc.page_content}" for i,doc in enumerate(retrieve_docs) ]
formated_docs = ".n".join(doc_lines)

In [12]:
doc_lines

[' 1 BMW is a German luxury automobile manufacturer founded in 1916.\n\nBMW is headquartered in Munich, Germany.\n\nBMW produces premium cars, motorcycles, and electric vehicles.\n\nThe BMW i4 and BMW iX are popular electric vehicle models.',
 ' 2 BMW continues to invest heavily in sustainable transportation technologies.\n\nMercedes-Benz is another German automotive company known for luxury vehicles.\n\nMercedes-Benz develops advanced driver assistance and autonomous driving systems.\n\nThe Mercedes EQS is a flagship electric luxury sedan.',
 " 3 Audi is a premium automobile manufacturer owned by the Volkswagen Group.\n\nAudi focuses on innovation, performance, and electric mobility.\n\nThe Audi e-tron series represents the company's electric vehicle lineup.\n\nTesla is an American electric vehicle company founded by Elon Musk.",
 ' 4 Tesla is known for its Model S, Model 3, Model X, and Model Y vehicles.\n\nTesla pioneered large-scale adoption of electric vehicles worldwide.\n\nGroq 

In [13]:
response=chain.invoke({"question":query,"documents":formated_docs})
response

'<think>\nOkay, let\'s tackle this. The user is asking "What is BMW?" So I need to find the documents that best answer that. \n\nLooking at document 1: It starts by stating BMW is a German luxury car manufacturer founded in 1916. It also mentions headquarters in Munich, and the products they make including electric vehicles. That seems very directly relevant.\n\nDocument 2 talks about BMW investing in sustainable tech, but then shifts to Mercedes-Benz and Tesla. While it\'s still about BMW a bit, the main focus is on competitors. Maybe not as relevant as document 1.\n\nDocument 3 is about Audi and Tesla. Not directly related to BMW, so probably lower relevance.\n\nDocument 4 focuses on Tesla\'s models and Groq. Definitely not about BMW.\n\nDocuments 5,6,7,8 are about AI and machine learning, which are unrelated to the question about BMW. \n\nSo the ranking should be 1 first, then 2, then maybe 3 and 4, but since they mention BMW in 2 a bit, maybe 2 is second. The rest are not relevant.